# Complete classifier analysis

Analysis-only pipeline for missing classifier metrics and visualizations. It never trains models, scores text, or changes source artifacts. Smoke mode uses one artifact and one row per analysis dimension; full mode removes those limits. Each step is resumable through a manifest.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.calibration import calibration_curve
from sklearn.metrics import ConfusionMatrixDisplay, precision_recall_curve, roc_curve

from notebooks.utils.classification_results_analysis_utils import load_classification_run, load_prediction_artifacts
from notebooks.utils.complete_classifier_analysis_utils import (
    AnalysisPaths, binary_metric_table, calibration_table, configuration_fingerprint,
    deduplicate_parent_counts, evaluation_metric_table, limit_smoke_rows, run_step,
    select_smoke_artifacts, unavailable_table,
)

DATA_ROOT = Path(os.environ.get("TEXT_DETECTION_DATA", "D:/Projects/master_diploma/text-detection-data"))
SOURCE_RUN_ID = os.environ.get("CLASSIFIER_ANALYSIS_SOURCE_RUN_ID", "feature-vector-classification-2026-09-25-logreg-random-forest-max600-min100")
ANALYSIS_RUN_ID = os.environ.get("CLASSIFIER_ANALYSIS_RUN_ID", f"complete-analysis-{SOURCE_RUN_ID}")
ANALYSIS_ROOT = DATA_ROOT / "experiment-results" / "classification" / "analysis"
SMOKE_MODE = False
SMOKE_ARTIFACT_LIMIT = 1
SMOKE_ROWS_PER_GROUP = 1
SMOKE_MODEL_FAMILY = os.environ.get("CLASSIFIER_ANALYSIS_MODEL_FAMILY", "logistic_regression")
SMOKE_TRAINING_WINDOW = os.environ.get("CLASSIFIER_ANALYSIS_TRAINING_WINDOW", "32")
CALIBRATION_BINS = 10
paths = AnalysisPaths.from_root(ANALYSIS_ROOT, ANALYSIS_RUN_ID)
INTERMEDIATE_ROOT = paths.intermediate
OUTPUT_ROOT = paths.output
paths.intermediate.mkdir(parents=True, exist_ok=True)
paths.output.mkdir(parents=True, exist_ok=True)
FIGURE_ROOT = paths.output / "figures"
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
def save_table(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)


def save_figure(figure: plt.Figure, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    figure.tight_layout()
    figure.savefig(path, dpi=160, bbox_inches="tight")
    plt.close(figure)


def empty_or_load_run() -> tuple[object | None, list[str]]:
    result_root = DATA_ROOT / "experiment-results" / "classification" / "evaluations" / SOURCE_RUN_ID
    if not SOURCE_RUN_ID or not result_root.is_dir():
        return None, [f"source result run is unavailable: {result_root}"]
    try:
        return load_classification_run(DATA_ROOT, run_id=SOURCE_RUN_ID), []
    except (OSError, ValueError) as error:
        return None, [f"source result run could not be loaded: {error}"]


results, warnings = empty_or_load_run()
tables = {} if results is None else dict(results.tables)
if results is not None:
    for optional_name in ("training_coverage", "training_metrics", "evaluation_predictions", "predictions"):
        optional_path = results.root / f"{optional_name}.csv"
        if optional_path.is_file():
            tables[optional_name] = pd.read_csv(optional_path)
    if "predictions" not in tables:
        nested_predictions = load_prediction_artifacts(
            DATA_ROOT,
            run_id=SOURCE_RUN_ID,
            progress_description="Loading prediction artifacts",
            columns=("label", "mg_score", "dataset", "partition"),
        )
        if not nested_predictions.empty:
            tables["predictions"] = nested_predictions

evaluation = tables.get("evaluation_metrics", pd.DataFrame()).copy()
if SMOKE_MODE and not evaluation.empty:
    if "model_family" in evaluation.columns:
        family_match = evaluation.loc[evaluation["model_family"].astype(str).eq(SMOKE_MODEL_FAMILY)]
        if not family_match.empty:
            evaluation = family_match
    if "training_window" in evaluation.columns:
        window_match = evaluation.loc[evaluation["training_window"].astype(str).eq(SMOKE_TRAINING_WINDOW)]
        if not window_match.empty:
            evaluation = window_match
    evaluation = select_smoke_artifacts(evaluation, limit=SMOKE_ARTIFACT_LIMIT)
run_fingerprint = configuration_fingerprint({
    "analysis_schema_version": 3,
    "source_run_id": SOURCE_RUN_ID,
    "smoke_mode": SMOKE_MODE,
    "smoke_artifact_limit": SMOKE_ARTIFACT_LIMIT,
    "smoke_rows_per_group": SMOKE_ROWS_PER_GROUP,
    "smoke_model_family": SMOKE_MODEL_FAMILY,
    "smoke_training_window": SMOKE_TRAINING_WINDOW,
})

## Artifact inventory and training composition

In [ ]:
inventory_path = paths.intermediate / "artifact_inventory.csv"
composition_path = paths.output / "training_composition.csv"

def build_inventory() -> None:
    rows = []
    for name, frame in tables.items():
        rows.append({"source_run_id": SOURCE_RUN_ID, "table": name, "rows": len(frame), "columns": "|".join(frame.columns)})
    save_table(pd.DataFrame(rows), inventory_path)

run_step(paths, step_id="artifact_inventory", fingerprint=run_fingerprint, expected_outputs=[inventory_path], callback=build_inventory)

def build_composition() -> None:
    coverage = tables.get("training_coverage", pd.DataFrame()).copy()
    if coverage.empty:
        save_table(pd.DataFrame(columns=["partition", "label", "row_count", "parent_count"]), composition_path)
        return
    label_column = "label" if "label" in coverage.columns else None
    parent_column = "text_id" if "text_id" in coverage.columns else None
    if label_column and parent_column:
        parent = deduplicate_parent_counts(coverage, parent_columns=[parent_column], label_column=label_column)
        rows = coverage.groupby(label_column, dropna=False).size().rename("row_count").reset_index()
        composition = rows.merge(parent, on=label_column, how="outer")
    else:
        composition = coverage.copy()
    save_table(composition, composition_path)

run_step(paths, step_id="training_composition", fingerprint=run_fingerprint, expected_outputs=[composition_path], callback=build_composition)

## Binary metrics, confusion matrices, calibration and score curves

In [ ]:
evaluation = evaluation if "evaluation" in globals() else tables.get("evaluation_metrics", pd.DataFrame()).copy()
predictions = tables.get("evaluation_predictions", pd.DataFrame()).copy()
if predictions.empty:
    predictions = tables.get("predictions", pd.DataFrame()).copy()
prediction_columns = {"label", "mg_score"}
prediction_unavailable_reason = "detailed prediction artifacts are unavailable"
if predictions.empty:
    warnings.append(prediction_unavailable_reason)
elif not prediction_columns.issubset(predictions.columns):
    prediction_unavailable_reason = f"detailed predictions are missing columns: {sorted(prediction_columns.difference(predictions.columns))}"
    warnings.append(prediction_unavailable_reason)
if SMOKE_MODE and not predictions.empty and {"dataset", "partition"}.issubset(predictions.columns):
    predictions = limit_smoke_rows(predictions, rows_per_group=SMOKE_ROWS_PER_GROUP, group_columns=["dataset", "partition"])

metric_path = paths.output / "binary_metrics.csv"
confusion_path = FIGURE_ROOT / "confusion_matrices.png"
curve_path = FIGURE_ROOT / "score_curves.png"

def build_binary_metrics() -> None:
    if not prediction_columns.issubset(predictions.columns):
        frame = evaluation_metric_table(evaluation) if not evaluation.empty else unavailable_table(prediction_unavailable_reason)
        save_table(frame, metric_path)
        return
    frame = binary_metric_table(predictions, label_column="label", score_column="mg_score")
    save_table(frame, metric_path)

run_step(paths, step_id="binary_metrics", fingerprint=run_fingerprint, expected_outputs=[metric_path], callback=build_binary_metrics)

def build_confusion_matrices() -> None:
    figure, axis = plt.subplots(figsize=(4.5, 4))
    if predictions.empty or not {"label", "mg_score"}.issubset(predictions.columns):
        axis.text(0.5, 0.5, "No binary predictions", ha="center", va="center")
        axis.set_axis_off()
    else:
        labels = predictions["label"].astype(int)
        predicted = (predictions["mg_score"].astype(float) >= 0.5).astype(int)
        ConfusionMatrixDisplay.from_predictions(labels, predicted, ax=axis, display_labels=["HG", "MG"])
        axis.set_title("Binary confusion matrix")
    save_figure(figure, confusion_path)

run_step(paths, step_id="confusion_matrices", fingerprint=run_fingerprint, expected_outputs=[confusion_path], callback=build_confusion_matrices)

def build_score_curves() -> None:
    figure, axes = plt.subplots(1, 2, figsize=(9, 4))
    if predictions.empty or predictions["label"].nunique() < 2 or not {"label", "mg_score"}.issubset(predictions.columns):
        for axis in axes:
            axis.text(0.5, 0.5, "Two-class predictions unavailable", ha="center", va="center")
            axis.set_axis_off()
    else:
        labels = predictions["label"].astype(int)
        scores = predictions["mg_score"].astype(float)
        fpr, tpr, _ = roc_curve(labels, scores)
        precision, recall, _ = precision_recall_curve(labels, scores)
        axes[0].plot(fpr, tpr); axes[0].set(xlabel="FPR", ylabel="TPR", title="ROC curve")
        axes[1].plot(recall, precision); axes[1].set(xlabel="Recall", ylabel="Precision", title="Precision-recall curve")
    save_figure(figure, curve_path)

run_step(paths, step_id="score_curves", fingerprint=run_fingerprint, expected_outputs=[curve_path], callback=build_score_curves)

## Calibration, thresholds and error slices

In [ ]:
calibration_path = paths.output / "calibration.csv"
reliability_path = FIGURE_ROOT / "reliability_diagram.png"
threshold_path = paths.output / "operating_points.csv"
threshold_figure_path = FIGURE_ROOT / "threshold_sweep.png"

def build_calibration() -> None:
    if not prediction_columns.issubset(predictions.columns):
        save_table(unavailable_table(prediction_unavailable_reason), calibration_path)
        return
    frame = calibration_table(predictions, score_column="mg_score", label_column="label", bins=CALIBRATION_BINS)
    save_table(frame, calibration_path)

run_step(paths, step_id="calibration", fingerprint=run_fingerprint, expected_outputs=[calibration_path], callback=build_calibration)

def build_reliability() -> None:
    figure, axis = plt.subplots(figsize=(5, 4))
    if predictions.empty or predictions["label"].nunique() < 2 or not {"label", "mg_score"}.issubset(predictions.columns):
        axis.text(0.5, 0.5, "Calibration unavailable", ha="center", va="center"); axis.set_axis_off()
    else:
        observed, predicted = calibration_curve(predictions["label"], predictions["mg_score"], n_bins=CALIBRATION_BINS, strategy="uniform")
        axis.plot(predicted, observed, marker="o", label="model"); axis.plot([0, 1], [0, 1], linestyle="--", label="perfect")
        axis.set(xlabel="Mean predicted MG score", ylabel="Observed MG rate", title="Reliability diagram"); axis.legend()
    save_figure(figure, reliability_path)

run_step(paths, step_id="reliability_diagram", fingerprint=run_fingerprint, expected_outputs=[reliability_path], callback=build_reliability)

def build_threshold_sweep() -> None:
    rows = []
    if not predictions.empty and {"label", "mg_score"}.issubset(predictions.columns):
        for threshold in tqdm(np.linspace(0.05, 0.95, 19), desc="Threshold sweep", unit="threshold"):
            metrics = binary_metric_table(predictions, label_column="label", score_column="mg_score", threshold=float(threshold))
            rows.append(metrics.assign(threshold=threshold))
    frame = pd.concat(rows, ignore_index=True) if rows else unavailable_table(prediction_unavailable_reason)
    save_table(frame, threshold_path)
    figure, axis = plt.subplots(figsize=(7, 4))
    if not frame.empty and "threshold" in frame.columns:
        for column in ("fpr", "tpr", "macro_f1", "precision", "recall"):
            axis.plot(frame["threshold"], frame[column], label=column)
        axis.set(xlabel="Threshold", ylabel="Metric", title="Threshold sweep"); axis.legend()
    else:
        axis.text(0.5, 0.5, "Threshold analysis unavailable", ha="center", va="center"); axis.set_axis_off()
    save_figure(figure, threshold_figure_path)

run_step(paths, step_id="threshold_sweep", fingerprint=run_fingerprint, expected_outputs=[threshold_path, threshold_figure_path], callback=build_threshold_sweep)

## Prefix, robustness and SemEval outputs

In [ ]:
def export_existing_table(step_id: str, table_name: str, output_name: str) -> None:
    output_path = paths.output / output_name
    def callback() -> None:
        frame = tables.get(table_name, pd.DataFrame()).copy()
        if frame.empty:
            frame = unavailable_table(f"source table is unavailable: {table_name}")
        if SMOKE_MODE and not frame.empty:
            groups = [column for column in ("dataset", "partition", "condition", "bounds", "training_window") if column in frame.columns]
            if groups:
                frame = limit_smoke_rows(frame, rows_per_group=SMOKE_ROWS_PER_GROUP, group_columns=groups)
        save_table(frame, output_path)
    fingerprint = configuration_fingerprint({"base": run_fingerprint, "step": step_id, "table": table_name})
    run_step(paths, step_id=step_id, fingerprint=fingerprint, expected_outputs=[output_path], callback=callback)

for step_id, table_name, output_name in tqdm((
    ("bounded_prefix", "bounded_prefix_metrics", "bounded_prefix_metrics.csv"),
    ("robustness", "robustness_metrics", "robustness_metrics.csv"),
    ("semeval_localization", "semeval_evaluations", "semeval_metrics.csv"),
    ("feature_and_window_summary", "evaluation_metrics", "feature_and_window_summary.csv"),
), desc="Exporting report tables", unit="table"):
    export_existing_table(step_id, table_name, output_name)

In [ ]:
def table_plot_step(step_id: str, table_name: str, figure_name: str, x_column: str, y_column: str) -> None:
    figure_path = FIGURE_ROOT / figure_name
    def callback() -> None:
        frame = tables.get(table_name, pd.DataFrame()).copy()
        figure, axis = plt.subplots(figsize=(7, 4))
        if frame.empty or x_column not in frame.columns or y_column not in frame.columns:
            axis.text(0.5, 0.5, f"{table_name} unavailable", ha="center", va="center"); axis.set_axis_off()
        else:
            frame = frame.dropna(subset=[x_column, y_column])
            axis.bar(frame[x_column].astype(str), pd.to_numeric(frame[y_column], errors="coerce"))
            axis.set(xlabel=x_column, ylabel=y_column, title=step_id); axis.tick_params(axis="x", rotation=45)
        save_figure(figure, figure_path)
    fingerprint = configuration_fingerprint({"base": run_fingerprint, "step": step_id, "table": table_name, "x": x_column, "y": y_column})
    run_step(paths, step_id=step_id, fingerprint=fingerprint, expected_outputs=[figure_path], callback=callback)

table_plot_step("training_composition_figure", "training_coverage", "training_composition.png", "split", "row_count")
table_plot_step("feature_window_summary", "evaluation_metrics", "feature_window_summary.png", "feature_set", "macro_f1")
table_plot_step("bounded_prefix_figure", "bounded_prefix_metrics", "bounded_prefix_quality.png", "bounds", "macro_f1")
table_plot_step("robustness_figure", "robustness_metrics", "robustness_conditions.png", "condition", "tpr")
table_plot_step("semeval_figure", "semeval_metrics", "semeval_tradeoff.png", "training_window", "mg_f1")

error_slice_path = FIGURE_ROOT / "error_slices.png"
def build_error_slices() -> None:
    figure, axis = plt.subplots(figsize=(6, 4))
    axis.text(0.5, 0.5, "Detailed prediction artifacts required for error slices", ha="center", va="center")
    axis.set_axis_off()
    save_figure(figure, error_slice_path)
run_step(paths, step_id="error_slices", fingerprint=run_fingerprint, expected_outputs=[error_slice_path], callback=build_error_slices)

summary_table_path = paths.output / "analysis_summary.csv"
def build_analysis_summary() -> None:
    manifest = json.loads(paths.manifest.read_text(encoding="utf-8")) if paths.manifest.exists() else {"steps": {}}
    rows = [{"step_id": step_id, "status": record.get("status"), "fingerprint": record.get("fingerprint")} for step_id, record in manifest.get("steps", {}).items()]
    save_table(pd.DataFrame(rows), summary_table_path)
run_step(paths, step_id="analysis_summary", fingerprint=run_fingerprint, expected_outputs=[summary_table_path], callback=build_analysis_summary)

summary_path = paths.output / "run_summary.json"
manifest = json.loads(paths.manifest.read_text(encoding="utf-8")) if paths.manifest.exists() else {"steps": {}}
summary = {
    "analysis_run_id": ANALYSIS_RUN_ID,
    "source_run_id": SOURCE_RUN_ID,
    "smoke_mode": SMOKE_MODE,
    "data_root": str(DATA_ROOT),
    "completed_steps": sorted(manifest.get("steps", {}).keys()),
    "warnings": warnings,
    "intermediate_root": str(paths.intermediate),
    "output_root": str(paths.output),
}
summary_path.write_text(json.dumps(summary, indent=2, sort_keys=True), encoding="utf-8")
print(json.dumps(summary, indent=2, ensure_ascii=False))

In [ ]:
summary_path = paths.output / "run_summary.json"
manifest = json.loads(paths.manifest.read_text(encoding="utf-8")) if paths.manifest.exists() else {"steps": {}}
summary = {
    "analysis_run_id": ANALYSIS_RUN_ID,
    "source_run_id": SOURCE_RUN_ID,
    "smoke_mode": SMOKE_MODE,
    "data_root": str(DATA_ROOT),
    "completed_steps": sorted(manifest.get("steps", {}).keys()),
    "warnings": warnings,
    "intermediate_root": str(paths.intermediate),
    "output_root": str(paths.output),
}
summary_path.write_text(json.dumps(summary, indent=2, sort_keys=True), encoding="utf-8")
print(json.dumps(summary, indent=2, ensure_ascii=False))